# 01 — Problem Framing & Ingestion — Video Game Sales (vgchartz, 1980–2016)

**Stages 0 and 1 of the Tier A pipeline** (`DOCS/STRUCTURE.md`). Stage 0 is a thinking step, so it
is written down here *before* any data is touched — the question, the stakeholder, the issue tree,
the SCR hypothesis and the ghost deck. Stage 1 then loads the file exactly as delivered, proves what
it is, and records every defect it carries *before* anyone cleans anything.

The data review (`DOCS/vgsales_data_review.md`) was written from a first pass over the file. This
notebook re-derives each of its claims from the raw CSV rather than trusting it — and two of them do
not reproduce (§1.4).

## 0 — Problem framing

### 0.1 The business question (one sentence)

> **Where in the physical console market does sales value concentrate — across titles, publishers,
> platforms, genres and regions — and how much of a release's outcome can a greenlight committee
> foresee from what it knows at launch?**

### 0.2 Stakeholder and decision

| | |
|---|---|
| **Primary stakeholder** | Head of Portfolio Strategy at a mid-size **third-party** publisher (no platform of its own) |
| **Decision this feeds** | How to construct the annual release slate: how many titles, which genre × platform cells, which regions to localise and market for, and how much weight a pre-launch "hit score" deserves in the greenlight meeting |
| **What "done" looks like** | (1) A quantified concentration profile the slate can be sized against; (2) a genre × region localisation map with a stability verdict; (3) a go / no-go on using a launch-metadata hit score as a greenlight input, with its precision at the top of the ranking stated |
| **Analysis type** | Rungs 1–2 (descriptive / diagnostic, **Path A**) for concentration and regions; rung 4 (predictive, **Path B**) for the hit score. **Path C is out of scope**: no treatment, no assignment mechanism, no counterfactual — nothing here is a causal claim |
| **Modality & grain** | Cross-sectional tabular. **One row = one game × platform release**, lifetime units in millions, four regions. Title-level views aggregate rows by `Name` and are reported alongside where the answer could differ |

### 0.3 Scope boundaries — explicitly **out**

- Revenue, price, margin, ROI — the file has units only.
- Anything after 2016, digital, mobile, free-to-play — the file cannot see them.
- "Why did game X sell?" — no quality, marketing, budget or review data; causal claims are out.
- Forecasting the 2026 market. The structural lessons are what transfer; the levels do not.

### 0.4 MECE issue tree

```
Where does value concentrate, and how foreseeable is it?
│
├── A. The record      — what can this file say about time at all?
│     A1 coverage by year (release counts vs units)       A2 right-censoring of recent years
│     A3 what drove the post-2011 decline in the file
│
├── B. Titles          — how unequal is the outcome of a release?
│     B1 distribution shape, Lorenz / Gini                B2 has concentration changed by era?
│     B3 row (game × platform) vs title unit
│
├── C. Owners          — who captures the value?
│     C1 publisher concentration (top-5, HHI) by era      C2 first-party vs third-party premium
│     C3 does the premium survive within a platform?
│
├── D. Regions         — where is the value, and does taste differ?
│     D1 regional mix by era                               D2 genre × region taste (JP / EU index)
│     D3 is the taste map stable across eras?
│
├── E. Products        — does *what* you make matter as much as *who* makes it?
│     E1 genre economics (median, hit rate)                E2 variance explained: genre vs platform vs publisher
│
└── F. Foreseeability  — can launch metadata pick hits?
      F1 time-split, title-grouped hit classifier          F2 which feature families carry it (ablation)
      F3 does it work for third-party releases specifically?
```

Branches are mutually exclusive by *object* (the record / the release / the owner / the market /
the product / the forecast) and together exhaust the business question.

### 0.5 SCR — Stage 0 draft (a hypothesis, to be revised in `06`)

| | Draft |
|---|---|
| **Situation** | Physical console games sold ~8.9 bn units across 1980–2016; the market is widely understood to be hit-driven. |
| **Complication** | A third-party publisher must commit budgets 2–3 years before launch, without knowing which releases will hit, and the recorded market appears to collapse after 2010. |
| **Resolution (hypothesis)** | Value is concentrated in a few titles and in platform holders' own games; regional taste is structural; launch metadata gives a useful triage but cannot pick hits — so the slate must be sized for a power law, not picked by genre. |

### 0.6 Ghost deck

```
SLIDE 1 — Governing Thought   [TBD after analysis]

PART I · The record
  "The post-2011 decline in this file is [coverage / censoring / real] — [quantified]"
  [Placeholder: releases & units per year, decomposed]

PART II · Where value concentrates
  "10% of releases earn ~60% of units, and the share is [rising / stable]"
  [Placeholder: Lorenz curve + top-decile share by era]
  "Platform holders' own games out-sell third-party games [x]× — even on the same platform"
  [Placeholder: first- vs third-party hit rate by platform]

PART III · Where taste differs
  "Japan buys role-playing games and not shooters — and that map is [stable] across eras"
  [Placeholder: genre × region index heatmap]

PART IV · What a greenlight committee can foresee
  "Launch metadata ranks hits [x]× better than chance; [publisher history] carries it"
  [Placeholder: PR curves, model vs baselines; ablation]

RECOMMENDATION — slate sizing · localisation map · how much weight a hit score gets
APPENDIX — cleaning log, tests, model card, limits
```

### Stage 0 — Gate Checklist

- [x] Business question is a single, unambiguous sentence
- [x] Issue tree has ≥ 2 levels of decomposition and is MECE
- [x] "Done" is defined as concrete outcomes (slate sizing, localisation map, go/no-go on a hit score)
- [x] Primary stakeholder named, decision context documented
- [x] Scope boundaries explicit (revenue, post-2016, digital, causal — all out)
- [x] SCR drafted · [x] Ghost deck exists with action-title placeholders · [x] Governing Thought drafted (tentative)

## 1 — Ingestion

### 1.1 Setup

In [1]:
import hashlib, json, shutil, sys, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

ROOT = Path.cwd()
while not (ROOT / "archive" / "vgsales.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ARCHIVE = ROOT / "archive"
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
REPORTS, FIGS, MODELS = ROOT / "reports", ROOT / "reports" / "figures", ROOT / "models"
for p in (RAW, PROC, REPORTS, FIGS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

KF_PATH = REPORTS / "_key_figures.json"


def _plain(v):
    if hasattr(v, "item"):
        return v.item()
    if isinstance(v, (list, tuple)):
        return [_plain(x) for x in v]
    return v


def kf_update(**kw):
    """Append to reports/_key_figures.json — the only place the report reads numbers from."""
    kf = json.loads(KF_PATH.read_text()) if KF_PATH.exists() else {}
    kf.update({k: _plain(v) for k, v in kw.items()})
    KF_PATH.write_text(json.dumps(kf, indent=1, sort_keys=True))
    print(f"  _key_figures.json <- {', '.join(kw)}  ({len(kf)} figures total)")

SRC = ARCHIVE / "vgsales.csv"
print(f"project root: {ROOT.name}")
print(f"source file : {SRC.relative_to(ROOT)}  ({SRC.stat().st_size:,} bytes)")

project root: Video Game Sales
source file : archive\vgsales.csv  (1,355,781 bytes)


### 1.2 Load exactly as delivered, preserve an immutable copy, log lineage

No type coercion, no renaming, no cleaning. The SHA-256 is recorded so any later run can prove it
is reading the same bytes.

In [2]:
raw = pd.read_csv(SRC, encoding="utf-8")
sha = hashlib.sha256(SRC.read_bytes()).hexdigest()

immutable = RAW / "vgsales_raw_immutable.csv"
shutil.copyfile(SRC, immutable)
assert hashlib.sha256(immutable.read_bytes()).hexdigest() == sha

lineage = pd.DataFrame([{
    "file": SRC.name, "source": "Kaggle 'Video Game Sales' (gregorut) — scraped from vgchartz.com",
    "pulled_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "rows": raw.shape[0], "cols": raw.shape[1], "bytes": SRC.stat().st_size, "sha256": sha,
}])
lineage.to_csv(RAW / "_lineage.csv", index=False)
print(lineage.T.to_string(header=False))
print()
print(raw.dtypes.to_string())
raw.head(5)

file                                                             vgsales.csv
source      Kaggle 'Video Game Sales' (gregorut) — scraped from vgchartz.com
pulled_utc                                                  2026-10-14 08:46
rows                                                                   16598
cols                                                                      11
bytes                                                                1355781
sha256      e2076095ffcae2a92dbc6de6ecbd54455ee034bb38da550b2dc385e9265d4ebe

Rank              int64
Name                str
Platform            str
Year            float64
Genre               str
Publisher           str
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


`Year` arrives as **float64**, not int — because it contains nulls. The review notes it "stored as
string in the source"; pandas resolves it to float on read. Either way it must be cast in `02`.

### 1.3 Schema contract

A written contract of what each column must be. Every check is run; failures are recorded, not
silently fixed.

In [3]:
REGIONS = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
EXPECTED = ["Rank", "Name", "Platform", "Year", "Genre", "Publisher", *REGIONS, "Global_Sales"]
GENRES = {"Action", "Adventure", "Fighting", "Misc", "Platform", "Puzzle", "Racing",
          "Role-Playing", "Shooter", "Simulation", "Sports", "Strategy"}

yr = raw["Year"]
region_sum = raw[REGIONS].sum(axis=1).round(2)
resid = (region_sum - raw["Global_Sales"]).round(2)

contract = [
    ("columns exactly as documented", list(raw.columns) == EXPECTED, f"{raw.shape[1]} columns"),
    ("Rank unique", raw["Rank"].is_unique, f"max {raw['Rank'].max():,} for {len(raw):,} rows"),
    ("Rank contiguous 1..n", raw["Rank"].max() == len(raw),
     f"missing ranks: {sorted(set(range(1, raw['Rank'].max() + 1)) - set(raw['Rank']))}"),
    ("Name non-null", raw["Name"].notna().all(), f"{raw['Name'].isna().sum()} null"),
    ("Platform non-null", raw["Platform"].notna().all(), f"{raw['Platform'].nunique()} platforms"),
    ("Genre in the 12 documented values", set(raw["Genre"]) == GENRES, f"{raw['Genre'].nunique()} genres"),
    ("Year non-null", yr.notna().all(), f"{yr.isna().sum()} null ({yr.isna().mean():.1%})"),
    ("Year within 1980–2016 (snapshot)", yr.dropna().between(1980, 2016).all(),
     f"{(yr > 2016).sum()} rows after 2016"),
    ("Publisher non-null and known", raw["Publisher"].notna().all() and (raw["Publisher"] != "Unknown").all(),
     f"{raw['Publisher'].isna().sum()} null + {(raw['Publisher'] == 'Unknown').sum()} 'Unknown'"),
    ("sales non-negative", (raw[REGIONS + ['Global_Sales']] >= 0).all().all(), "min 0"),
    ("Global > 0.10 (documented > 100k copies)", (raw["Global_Sales"] > 0.1).all(),
     f"min {raw['Global_Sales'].min()}; {(raw['Global_Sales'] < 0.1).sum():,} rows < 0.10"),
    ("Global == sum of regions (exact)", (resid == 0).all(),
     f"{(resid != 0).sum():,} rows differ, max |Δ| {resid.abs().max()}"),
    ("Global == sum of regions (±0.02)", (resid.abs() <= 0.02).all(), "rounding tolerance"),
    ("Rank monotone in Global_Sales", raw.sort_values("Rank")["Global_Sales"].is_monotonic_decreasing,
     "Rank is a sorted index"),
]
schema = pd.DataFrame(contract, columns=["check", "pass", "detail"])
schema.to_csv(RAW / "_schema_validation.csv", index=False)
print(f"{schema['pass'].sum()} of {len(schema)} contract checks pass\n")
schema

8 of 14 contract checks pass



,check,pass,detail
0,columns exactly as documented,True,11 columns
1,Rank unique,True,"max 16,600 for 16,598 rows"
2,Rank contiguous 1..n,False,"missing ranks: [654, 14200]"
3,Name non-null,True,0 null
4,Platform non-null,True,31 platforms
5,Genre in the 12 documented values,True,12 genres
6,Year non-null,False,271 null (1.6%)
7,Year within 1980–2016 (snapshot),False,4 rows after 2016
8,Publisher non-null and known,False,58 null + 203 'Unknown'
9,sales non-negative,True,min 0


Six of fourteen checks fail. None is fatal; each becomes a line in the `02` cleaning log. Two of them
deserve a closer look because they contradict the data review.

### 1.4 Re-deriving the data review — two claims do not reproduce

In [4]:
print("Global_Sales minus sum of the four regions (rounded to 0.01):")
print(resid.value_counts().sort_index().to_string())
n_resid = int((resid != 0).sum())
n_resid_2 = int((resid.abs() == 0.02).sum())
print(f"\nrows that differ at all : {n_resid:,} ({n_resid/len(raw):.1%})")
print(f"rows that differ by 0.02: {n_resid_2}")

dup_full = raw.duplicated().sum()
dup_norank = raw.drop(columns="Rank").duplicated(keep=False)
print(f"\nfull-row duplicates incl. Rank: {dup_full}")
print(f"full-row duplicates excl. Rank: {raw.drop(columns='Rank').duplicated().sum()}")
print(raw[dup_norank].to_string())

Global_Sales minus sum of the four regions (rounded to 0.01):
-0.02        6
-0.01     2478
 0.00    12087
 0.01     2023
 0.02        4

rows that differ at all : 4,511 (27.2%)
rows that differ by 0.02: 10

full-row duplicates incl. Rank: 0
full-row duplicates excl. Rank: 1
        Rank                         Name Platform  Year    Genre Publisher  NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales
14997  15000  Wii de Asobu: Metroid Prime      Wii   NaN  Shooter  Nintendo       0.0       0.0      0.02          0.0          0.02
14999  15002  Wii de Asobu: Metroid Prime      Wii   NaN  Shooter  Nintendo       0.0       0.0      0.02          0.0          0.02


| Review claim (`vgsales_data_review.md` §2) | Reproduced? | What the file says |
|---|---|---|
| "Global vs sum of regions: only **10 rows** differ, max 0.02" | **No** | **4,511 rows (27%)** differ by ±0.01; 10 differ by ±0.02. Still rounding — but any validation asserting `Global == ΣRegions` exactly will flag a quarter of the file. Tolerance must be 0.02 |
| "**0** full-row duplicates" | **Only because of `Rank`** | `Rank` is a sorted index, so it makes every row unique. Excluding it, *Wii de Asobu: Metroid Prime* (ranks 15000 and 15002) is an exact duplicate |

Every other figure in the review reproduces to the stated precision (§1.6 checks them in bulk).

### 1.5 Integrity audit — repeated keys, impossible years, the "threshold" claim

In [5]:
rep = raw[raw.duplicated(["Name", "Platform"], keep=False)].sort_values(["Name", "Rank"])
print(f"repeated Name + Platform pairs: {rep.groupby(['Name', 'Platform']).ngroups}  ({len(rep)} rows)\n")
print(rep[["Rank", "Name", "Platform", "Year", "Publisher", *REGIONS, "Global_Sales"]].to_string(index=False))

repeated Name + Platform pairs: 5  (10 rows)

 Rank                        Name Platform   Year       Publisher  NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales
  604               Madden NFL 13      PS3 2012.0 Electronic Arts      2.11      0.23      0.00         0.22          2.56
16130               Madden NFL 13      PS3 2012.0 Electronic Arts      0.00      0.01      0.00         0.00          0.01
 1175 Need for Speed: Most Wanted     X360 2012.0 Electronic Arts      0.62      0.80      0.01         0.15          1.58
 1532 Need for Speed: Most Wanted     X360 2005.0 Electronic Arts      1.00      0.17      0.02         0.10          1.29
 5902 Need for Speed: Most Wanted       PC 2005.0 Electronic Arts      0.02      0.23      0.00         0.05          0.30
11678 Need for Speed: Most Wanted       PC 2012.0 Electronic Arts      0.00      0.06      0.00         0.02          0.08
 1717          Sonic the Hedgehog      PS3 2006.0            Sega      0.41      0.07      0.

Five repeated keys, four different stories — so they cannot share one rule:

| Key | Verdict | Handling in `02` |
|---|---|---|
| *Need for Speed: Most Wanted* × X360, PC | **Two different games** (2005 original, 2012 reboot) | keep both |
| *Wii de Asobu: Metroid Prime* × Wii | **Exact duplicate** (only `Rank` differs) | drop rank 15002 |
| *Madden NFL 13* × PS3 | 2.56 M row plus a 0.01 M stray with the same year and publisher | drop the stray |
| *Sonic the Hedgehog* × PS3 | 1.18 M row plus a 0.48 M **EU-only** row with no year or publisher | **merge**, not drop — the stray carries real EU units the main row lacks |

In [6]:
# Release year vs the platform's launch year. Launch years are external reference facts (first
# launch in any region), NOT the data's own minimum — the data's minimum is what we are testing.
PLATFORM_LAUNCH = {
    "2600": 1977, "NES": 1983, "SNES": 1990, "N64": 1996, "GC": 2001, "Wii": 2006, "WiiU": 2012,
    "GB": 1989, "GBA": 2001, "DS": 2004, "3DS": 2011, "PS": 1994, "PS2": 2000, "PS3": 2006,
    "PS4": 2013, "PSP": 2004, "PSV": 2011, "XB": 2001, "X360": 2005, "XOne": 2013, "GEN": 1988,
    "SAT": 1994, "DC": 1998, "GG": 1990, "SCD": 1991, "NG": 1990, "3DO": 1993, "TG16": 1987,
    "PCFX": 1994, "WS": 1999, "PC": None,
}
assert set(PLATFORM_LAUNCH) == set(raw["Platform"]), "unmapped platform"
launch = raw["Platform"].map(PLATFORM_LAUNCH)
pre_launch = raw[launch.notna() & yr.notna() & (yr < launch)]
print(f"rows released BEFORE their platform existed: {len(pre_launch)}")
print(pre_launch[["Rank", "Name", "Platform", "Year", "Publisher", "Global_Sales"]].to_string(index=False))
print(f"\nrows dated after the 2016 snapshot: {(yr > 2016).sum()}")
print(raw.loc[yr > 2016, ["Rank", "Name", "Platform", "Year", "Global_Sales"]].to_string(index=False))

rows released BEFORE their platform existed: 4
 Rank                                   Name Platform   Year                    Publisher  Global_Sales
 1352                     Disney's DuckTales       GB 1988.0                       Capcom          1.43
 2085                         NFL Fever 2002       XB 2000.0       Microsoft Game Studios          0.99
12272 ESPN Winter X-Games: Snowboarding 2002      GBA 2000.0 Konami Digital Entertainment          0.06
15870    Strongest Tokyo University Shogi DS       DS 1985.0                        Mycom          0.02

rows dated after the 2016 snapshot: 4
 Rank                                             Name Platform   Year  Global_Sales
 5959                           Imagine: Makeup Artist       DS 2020.0          0.29
14393 Phantasy Star Online 2 Episode 4: Deluxe Package      PS4 2017.0          0.03
16244 Phantasy Star Online 2 Episode 4: Deluxe Package      PSV 2017.0          0.01
16441                 Brothers Conflict: Precious Baby

In [7]:
g = raw["Global_Sales"]
print(f"min Global_Sales            : {g.min()}  (= 10,000 units)")
print(f"rows below 0.10 (100k units): {(g < 0.10).sum():,}  ({(g < 0.10).mean():.1%})")
print(f"rows below 0.05             : {(g < 0.05).sum():,}  ({(g < 0.05).mean():.1%})")
print(f"rows exactly 0.01           : {(g == 0.01).sum():,}")
print("\nzeros by region (no release in region, or rounded below 5k units — indistinguishable):")
for c in REGIONS:
    print(f"   {c:12s} {(raw[c] == 0).sum():>6,}  ({(raw[c] == 0).mean():.0%})")

min Global_Sales            : 0.01  (= 10,000 units)


rows below 0.10 (100k units): 5,781  (34.8%)
rows below 0.05             : 3,145  (18.9%)
rows exactly 0.01           : 618

zeros by region (no release in region, or rounded below 5k units — indistinguishable):
   NA_Sales      4,499  (27%)
   EU_Sales      5,730  (35%)
   JP_Sales     10,455  (63%)
   Other_Sales   6,477  (39%)


Four rows predate their platform's launch (a DS game in 1985, a Game Boy game in 1988, two
"2002"-branded sports titles dated 2000 on platforms launched in 2001) and four postdate the snapshot
(including a DS game in 2020). All eight years are set to missing in `02`.

The dataset description says it lists "games with sales greater than 100,000 copies". **5,781 rows
(34.8%) are below that line**, the smallest at 10,000 units. The threshold is not a property of the
file, so nothing downstream may assume the tail is truncated at 100k — and conversely, *the floor of
the tail is wherever vgchartz chose to stop cataloguing*, which matters for any count-by-year claim
(`03` §A).

### 1.6 Bulk re-derivation of the review's headline figures

In [8]:
def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))


s = g.sort_values(ascending=False).to_numpy()
reg = raw[REGIONS].sum()
checks = pd.DataFrame([
    ("total global units (M)", "8,920", f"{g.sum():,.0f}"),
    ("median / mean per row (M)", "0.17 / 0.54", f"{g.median():.2f} / {g.mean():.2f}"),
    ("skew", "17.4", f"{g.skew():.1f}"),
    ("top 1% of rows, share of units", "21%", f"{s[:len(s)//100].sum()/s.sum():.1%}"),
    ("top 10% of rows, share of units", "59%", f"{s[:len(s)//10].sum()/s.sum():.1%}"),
    ("rows ≥ 1 M / ≥ 10 M", "2,081 / 62", f"{(g>=1).sum():,} / {(g>=10).sum()}"),
    ("unique titles / platforms / genres / publishers", "11,493 / 31 / 12 / 578",
     f"{raw.Name.nunique():,} / {raw.Platform.nunique()} / {raw.Genre.nunique()} / {raw.Publisher.nunique()}"),
    ("titles on > 1 platform", "2,774", f"{(raw.groupby('Name').Platform.nunique() > 1).sum():,}"),
    ("region split NA/EU/JP/Other", "49.2/27.3/14.5/8.9",
     "/".join(f"{v:.1%}".rstrip('%') for v in reg / reg.sum())),
    ("rows with JP = 0", "63%", f"{(raw.JP_Sales == 0).mean():.0%}"),
], columns=["figure", "review says", "file says"])
checks

,figure,review says,file says
0,total global units (M),"8,920","8,920"
1,median / mean per row (M),0.17 / 0.54,0.17 / 0.54
2,skew,17.4,17.4
3,"top 1% of rows, share of units",21%,21.0%
4,"top 10% of rows, share of units",59%,58.8%
5,rows ≥ 1 M / ≥ 10 M,"2,081 / 62","2,081 / 62"
6,unique titles / platforms / genres / publishers,"11,493 / 31 / 12 / 578","11,493 / 31 / 12 / 578"
7,titles on > 1 platform,"2,774","2,774"
8,region split NA/EU/JP/Other,49.2/27.3/14.5/8.9,49.3/27.3/14.5/8.9
9,rows with JP = 0,63%,63%


All reproduce. (NA is 49.3%, not 49.2% — a rounding difference.)

### 1.7 Governance

| Item | Assessment |
|---|---|
| PII | None. Rows are commercial products; publishers are companies |
| Licence / purpose | Public Kaggle dataset of a public website scrape; used for non-commercial portfolio analysis |
| Measurement | vgchartz figures are **modelled estimates**, not audited sell-through, and include console-bundled copies (e.g. *Wii Sports*). Every claim is about *this record of the market*, and is labelled as such |
| Fairness audit | Not applicable in the person-level sense — the hit model scores products, not people. The equivalent check (does the score work for third-party publishers, not just platform holders?) is run in `05` |

In [9]:
pd.DataFrame([
    ("PII present", "no"), ("licence", "public Kaggle scrape — portfolio use"),
    ("measurement", "vgchartz estimates incl. bundles; not audited"),
    ("person-level fairness audit", "n/a — products, not people; publisher-type parity checked in 05"),
], columns=["item", "assessment"]).to_csv(RAW / "_governance.csv", index=False)

kf_update(
    n_rows_raw=len(raw), n_cols_raw=raw.shape[1], sha256=sha,
    n_titles_raw=raw.Name.nunique(), n_platforms=raw.Platform.nunique(), n_genres=raw.Genre.nunique(),
    n_publishers_raw=raw.Publisher.nunique(),
    total_units_m=float(g.sum()), median_units_m=float(g.median()), mean_units_m=float(g.mean()),
    skew_units=float(g.skew()),
    top1_row_share=float(s[:len(s)//100].sum()/s.sum()), top10_row_share=float(s[:len(s)//10].sum()/s.sum()),
    gini_rows_raw=gini(g),
    n_hits_raw=int((g >= 1).sum()), n_blockbusters_raw=int((g >= 10).sum()),
    region_share_na=float(reg.NA_Sales/reg.sum()), region_share_eu=float(reg.EU_Sales/reg.sum()),
    region_share_jp=float(reg.JP_Sales/reg.sum()), region_share_other=float(reg.Other_Sales/reg.sum()),
    n_year_missing_raw=int(yr.isna().sum()),
    n_pub_missing_raw=int(raw.Publisher.isna().sum()), n_pub_unknown_raw=int((raw.Publisher == "Unknown").sum()),
    n_region_sum_mismatch=n_resid, n_region_sum_mismatch_002=n_resid_2,
    share_region_sum_mismatch=n_resid/len(raw),
    n_below_100k=int((g < 0.10).sum()), share_below_100k=float((g < 0.10).mean()),
    n_pre_launch_rows=len(pre_launch), n_post_snapshot_rows=int((yr > 2016).sum()),
    n_repeated_keys=int(rep.groupby(['Name', 'Platform']).ngroups),
    n_schema_checks=len(schema), n_schema_pass=int(schema["pass"].sum()),
)

  _key_figures.json <- n_rows_raw, n_cols_raw, sha256, n_titles_raw, n_platforms, n_genres, n_publishers_raw, total_units_m, median_units_m, mean_units_m, skew_units, top1_row_share, top10_row_share, gini_rows_raw, n_hits_raw, n_blockbusters_raw, region_share_na, region_share_eu, region_share_jp, region_share_other, n_year_missing_raw, n_pub_missing_raw, n_pub_unknown_raw, n_region_sum_mismatch, n_region_sum_mismatch_002, share_region_sum_mismatch, n_below_100k, share_below_100k, n_pre_launch_rows, n_post_snapshot_rows, n_repeated_keys, n_schema_checks, n_schema_pass  (33 figures total)


## Stage 1 — Gate Checklist

- [x] Raw data saved to `data/raw/vgsales_raw_immutable.csv`, hash-verified
- [x] Source, timestamp, row/column counts and SHA-256 logged (`data/raw/_lineage.csv`)
- [x] Schema checked against a written contract; 6 of 14 failures documented, none silently fixed
- [x] Governance review recorded (`data/raw/_governance.csv`) — no PII
- [x] Data review re-derived from the raw file: all headline figures reproduce; **two claims do not**
      (region-sum reconciliation, duplicate count) and are corrected here

**Carried into `02`:** 4 repeated keys with 4 different handling rules · pre-launch and post-snapshot
years · NaN vs "Unknown" publishers · a ±0.02 reconciliation tolerance · no 100k threshold.